# Telco Customer Churn Prediction\nDataset: IBM Telco Customer Churn Dataset from Kaggle.\n

In [ ]:
# Telco Customer Churn Prediction
# Dataset: IBM Telco Customer Churn Dataset from Kaggle
# CSV file needed: WA_Fn-UseC_-Telco-Customer-Churn.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 1. Load dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset shape:", df.shape)
print(df.head())


# 2. Data understanding
print("\nDataset information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# 3. Data cleaning
# TotalCharges has blank values, so we convert it to numeric.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

print("\nMissing values after cleaning TotalCharges:")
print(df["TotalCharges"].isnull().sum())


# 4. Exploratory Data Analysis

# Churn distribution
churn_counts = df["Churn"].value_counts()

plt.figure(figsize=(6, 4))
churn_counts.plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

print("\nChurn distribution:")
print(churn_counts)


# Churn by contract type
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100

contract_churn.plot(kind="bar", figsize=(7, 4))
plt.title("Churn Percentage by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Percentage")
plt.xticks(rotation=30)
plt.show()

print("\nChurn percentage by contract type:")
print(contract_churn)


# Average numerical values by churn
avg_churn = df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]].mean()

print("\nAverage values by churn:")
print(avg_churn)


# 5. Prepare data for machine learning
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("\nCategorical features:", categorical_features)
print("Numerical features:", numerical_features)


# 6. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# 7. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


# 8. Logistic Regression
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)

print("\nLogistic Regression Accuracy:")
print(accuracy_score(y_test, logistic_pred))

print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, logistic_pred))


# 9. Random Forest
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=150, random_state=42))
])

random_forest_model.fit(X_train, y_train)
rf_pred = random_forest_model.predict(X_test)

print("\nRandom Forest Accuracy:")
print(accuracy_score(y_test, rf_pred))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_pred))


# 10. Confusion Matrix
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.xticks([0, 1], ["No Churn", "Churn"])
plt.yticks([0, 1], ["No Churn", "Churn"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

print("\nConfusion matrix:")
print(cm)


# 11. Feature Importance
encoded_cat_features = random_forest_model.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_features)
all_features = numerical_features + list(encoded_cat_features)

importances = random_forest_model.named_steps["classifier"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": all_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 10 important features:")
print(feature_importance.head(10))

feature_importance.head(10).plot(
    kind="barh",
    x="Feature",
    y="Importance",
    legend=False,
    figsize=(8, 5)
)

plt.title("Top 10 Important Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.show()


# 12. Business Insights
print("\nBusiness Insights:")
print("1. Customers with month-to-month contracts are more likely to churn.")
print("2. Customers with shorter tenure have higher churn risk.")
print("3. Customers with higher monthly charges may have higher churn risk.")
print("4. The company can offer promotions, better support, or contract upgrades to reduce churn.")
